In [1]:
import pandas as pd
import torch 
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import torch.optim as optim
import numpy as np
import sklearn
from sklearn.preprocessing import StandardScaler

torch.cuda.empty_cache()

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
device = torch.device('cpu')

In [ ]:
## ***** PREPROCESSING DATA booooooooorrriiiingggg😒😒🙄 ***** ##

def preprocess(df, is_train=True):
    df = df.drop(columns=['PassengerId', 'Name'])

    # fill null rows with the median instead of dropping
    non_binary = [col for col in df.columns if df[col].nunique() > 2 and df[col].dtype in ['float64', 'int64']]
    for col in non_binary:
        df[col] = df[col].fillna(df[col].median())

    # fill categorical rows with the mode
    binary_cols = [col for col in df.columns if df[col].nunique() == 2]
    for col in binary_cols:
        df[col] = df[col].fillna(df[col].mode()[0])

    cat_cols = ['HomePlanet', 'Destination', 'Cabin']
    for col in cat_cols:
        df[col] = df[col].fillna(df[col].mode()[0])

    # only drop nulls for train target column
    if is_train:
        df.dropna(subset=['Transported'], inplace=True)

    df[['Cabin_Deck', 'Cabin_Num', 'Cabin_Side']] = df['Cabin'].str.split('/', expand=True)
    df = df.drop(columns=['Cabin'])
    df['Cabin_Num'] = df['Cabin_Num'].astype(int)

    df[["CryoSleep", "VIP"]] = df[["CryoSleep", "VIP"]].astype(int)

    df = pd.get_dummies(df, columns=["HomePlanet", "Destination", "Cabin_Deck", "Cabin_Side"])

    dummy_cols = [col for col in df.columns if df[col].dtype == 'bool']
    df[dummy_cols] = df[dummy_cols].astype(int)

    non_binary = [col for col in df.columns if df[col].nunique() > 2 and df[col].dtype in ['float64', 'int64']]
    
    if is_train:
        df[non_binary] = scaler.fit_transform(df[non_binary])  # fit AND transform
    else:
        df[non_binary] = scaler.transform(df[non_binary])  # only transform

    return df

scaler = StandardScaler()
df_train = preprocess(pd.read_csv('train.csv'), is_train=True)
df_test = preprocess(pd.read_csv('test.csv'), is_train=False)
df_train

C:\Users\Aarul Aggarwal\AppData\Local\Temp\ipykernel_14524\1814811680.py:13: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[col] = df[col].fillna(df[col].mode()[0])
C:\Users\Aarul Aggarwal\AppData\Local\Temp\ipykernel_14524\1814811680.py:13: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[col] = df[col].fillna(df[col].mode()[0])


,CryoSleep,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Transported,Cabin_Num,...,Cabin_Deck_A,Cabin_Deck_B,Cabin_Deck_C,Cabin_Deck_D,Cabin_Deck_E,Cabin_Deck_F,Cabin_Deck_G,Cabin_Deck_T,Cabin_Side_P,Cabin_Side_S
0,0,0.711945,0,-0.333105,-0.281027,-0.283579,-0.270626,-0.263003,0,-1.191744,...,0,1,0,0,0,0,0,0,1,0
1,0,-0.334037,0,-0.168073,-0.275387,-0.241771,0.217158,-0.224205,1,-1.191744,...,0,0,0,0,0,1,0,0,0,1
2,0,2.036857,1,-0.268001,1.959998,-0.283579,5.695623,-0.219796,0,-1.191744,...,1,0,0,0,0,0,0,0,0,1
3,0,0.293552,0,-0.333105,0.523010,0.336851,2.687176,-0.092818,0,-1.191744,...,1,0,0,0,0,0,0,0,0,1
4,0,-0.891895,0,0.125652,-0.237159,-0.031059,0.231374,-0.261240,1,-1.189769,...,0,0,0,0,0,1,0,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8688,0,0.851410,1,-0.333105,3.992336,-0.283579,1.189173,-0.197751,0,-0.998198,...,1,0,0,0,0,0,0,0,1,0
8689,1,-0.752431,0,-0.333105,-0.281027,-0.283579,-0.270626,-0.263003,0,1.768722,...,0,0,0,0,0,0,1,0,0,1
8690,0,-0.194573,0,-0.333105,-0.281027,2.846999,-0.269737,-0.263003,1,1.770697,...,0,0,0,0,0,0,1,0,0,1
8691,0,0.223820,0,-0.333105,0.376365,-0.283579,0.043013,2.589576,0,0.009032,...,0,0,0,0,1,0,0,0,0,1


In [3]:
X_train_pre = df_train.drop(columns=['Transported']).values
Y_train_pre = df_train['Transported'].values

In [4]:
X_train = torch.tensor(X_train_pre, dtype=torch.float32).to(device)
Y_train = torch.tensor(Y_train_pre, dtype=torch.float32).to(device)
print(torch.cuda.is_available())  

#can skip this since it a comparatively small dataset but it is a good habit to manage it in batches for training faster and also shuffle the datapoints --
# --for each epoch preventing the model to learn order patterns
dataset = TensorDataset(X_train, Y_train)
dataloader = DataLoader(dataset, batch_size=16, shuffle=True)

Y_train

True


tensor([0., 1., 0.,  ..., 1., 0., 1.])

In [ ]:
## *** LEESSS GOOOO 💥💥🦅 *** ##

class Model(nn.Module):
    def __init__(self, input_size):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_size, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            
            nn.Linear(32, 1),
            nn.Sigmoid()

            # nn.Linear(input_size, 256),
            # nn.BatchNorm1d(256),
            # nn.LeakyReLU(0.01),
            # nn.Dropout(0.3),

            # nn.Linear(256, 128),
            # nn.BatchNorm1d(128),
            # nn.LeakyReLU(0.01),
            # nn.Dropout(0.3),

            # nn.Linear(128, 64),
            # nn.BatchNorm1d(64),
            # nn.LeakyReLU(0.01),
            # nn.Dropout(0.2),

            # nn.Linear(64, 32),
            # nn.LeakyReLU(0.01),

            # nn.Linear(32, 1),
            # nn.Sigmoid()
        )

    def forward(self, x):
        return self.network(x)
    
input_size = X_train.shape[1]
model = Model(input_size).to(device)

In [ ]:
## *** LOCK-IN ✌️ *** ##

criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.0007)

epochs=400

for epoch in range(epochs):
    model.train()
    total_loss=0
    correct=0
    total=0

    for X_batch, Y_batch in dataloader:
        X_batch = X_batch.to(device)
        Y_batch = Y_batch.to(device)

        output = model(X_batch).squeeze()
        loss = criterion(output, Y_batch)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item()

        predicted = (output > 0.5).float()
        correct += (predicted == Y_batch).sum().item()
        total += Y_batch.size(0)

    if (epoch%20 == 0):
            avgloss = total_loss/len(dataloader)
            accuracy = correct / total * 100
            print(f'Epoch {epoch}/{epochs}, Loss: {avgloss:.4f}, Accuracy: {accuracy:.2f}%')


Epoch 0/400, Loss: 0.5119, Accuracy: 75.27%
Epoch 20/400, Loss: 0.4150, Accuracy: 80.05%
Epoch 40/400, Loss: 0.4053, Accuracy: 80.50%
Epoch 60/400, Loss: 0.3898, Accuracy: 81.02%
Epoch 80/400, Loss: 0.3913, Accuracy: 80.87%
Epoch 100/400, Loss: 0.3900, Accuracy: 80.87%
Epoch 120/400, Loss: 0.3843, Accuracy: 81.02%
Epoch 140/400, Loss: 0.3816, Accuracy: 81.46%
Epoch 160/400, Loss: 0.3819, Accuracy: 81.32%
Epoch 180/400, Loss: 0.3808, Accuracy: 81.71%
Epoch 200/400, Loss: 0.3766, Accuracy: 81.41%
Epoch 220/400, Loss: 0.3714, Accuracy: 81.87%
Epoch 240/400, Loss: 0.3743, Accuracy: 81.48%
Epoch 260/400, Loss: 0.3750, Accuracy: 81.72%
Epoch 280/400, Loss: 0.3704, Accuracy: 81.54%
Epoch 300/400, Loss: 0.3743, Accuracy: 81.36%
Epoch 320/400, Loss: 0.3675, Accuracy: 82.20%
Epoch 340/400, Loss: 0.3729, Accuracy: 81.77%
Epoch 360/400, Loss: 0.3688, Accuracy: 82.05%
Epoch 380/400, Loss: 0.3677, Accuracy: 81.67%


In [7]:
passenger_ids = pd.read_csv('test.csv')['PassengerId']

model.eval()
with torch.no_grad():
    X_test_tensor = torch.tensor(df_test.values.astype('float32'), dtype=torch.float32).to(device)
    predictions = model(X_test_tensor).squeeze()
    predictions = (predictions > 0.5).cpu().numpy().astype(bool)

df_submit = pd.DataFrame({
    'PassengerId': passenger_ids,
    'Transported': predictions
})

df_submit.to_csv('submission.csv', index=False)
print(df_submit.head())
print(df_submit.shape)

  PassengerId  Transported
0     0013_01         True
1     0018_01        False
2     0019_01         True
3     0021_01         True
4     0023_01        False
(4277, 2)
